In [2]:
# 11.2
from sympy import *

L = 7
x, w, E, I, C1, C2 = symbols('x w E I C1 C2')

# 第一段方程及导数
y1 = (w/(E*I)) * (-3*L**2*x**2/16 + L*x**3/12)
dy1 = diff(y1, x)

# 第二段弯矩 (直接套负弯矩符号约定)
M2 = -w*(L-x)**2/2

# 积分两次
dy2 = integrate(M2/(E*I), x) + C1
y2 = integrate(dy2, x) + C2

# 在 L/2 处的边界连续条件 (相减等于0)
eq1 = dy1.subs(x, L/2) - dy2.subs(x, L/2)
eq2 = y1.subs(x, L/2) - y2.subs(x, L/2)

# 直接解出 C1, C2
ans = solve([eq1, eq2], [C1, C2])

# 代入答案，除以 w/EI 后展开
y2_ans = expand(y2.subs(ans) / (w/(E*I)))

print(y2_ans)

-x**4/24 + 7*x**3/6 - 49*x**2/4 + 7.14583333333333*x - 6.25260416666667


In [3]:
# 11.3
from sympy import *

x, C1, C2, C3, C4 = symbols('x C1 C2 C3 C4')

# 1. 输入参数
L = 7
P = 8
M = 8
EI = (174 * 10**6) * (48 * 10**-6) 

# 2. 第一段弯矩与积分 (注意这里 M 前面改成了减号！)
M1 = P*(L - x) - M
dy1 = integrate(M1/EI, x) + C1
y1 = integrate(dy1, x) + C2

# 3. 第二段弯矩与积分
M2 = 0
dy2 = integrate(M2/EI, x) + C3
y2 = integrate(dy2, x) + C4

# 4. 边界与连续性条件
ans = solve([
    dy1.subs(x, 0), 
    y1.subs(x, 0),
    dy1.subs(x, L) - dy2.subs(x, L),
    y1.subs(x, L) - y2.subs(x, L)
], [C1, C2, C3, C4])

# 5. 代入展开
y1_ans = expand(y1.subs(ans))
y2_ans = expand(y2.subs(ans))

# 提取并打印系数
def print_coeffs(expr, name):
    c = Poly(expr, x).all_coeffs()
    c.reverse()
    c = (c + [0]*5)[:5]
    print(f"=== {name} ===")
    for i in range(5):
        print(f"x^{i} = {float(c[i]):.10f}")

print_coeffs(y1_ans, "y1(x)")
print_coeffs(y2_ans, "y2(x)")

=== y1(x) ===
x^0 = 0.0000000000
x^1 = 0.0000000000
x^2 = 0.0028735632
x^3 = -0.0001596424
x^4 = 0.0000000000
=== y2(x) ===
x^0 = -0.0312899106
x^1 = 0.0167624521
x^2 = 0.0000000000
x^3 = 0.0000000000
x^4 = 0.0000000000


In [4]:
# 11.4
from sympy import *

# 1. 统一单位输入 (kN, m)
M = 65 
L = 1.912 
delta = 0.003 
E = 146 * 10**6  # GPa 转换为 kPa (kN/m^2)
I = 128.079468 * 10**-6  # mm^4 转换为 m^4

EI = E * I
F = symbols('F')

# 2. 叠加法推导自由端挠度
# 弯矩 M 引起的向下挠度 (跨度2L，力偶位于L处)
delta_M = (M * L**2) / (2 * EI) + (M * L / EI) * L 

# 支座反力 F 引起的向上挠度 (作用在全长 2L 的自由端)
delta_F = F * (2*L)**3 / (3 * EI)

eq = delta_M - delta_F - delta

ans = solve(eq, F)
print(f"|F| = {float(ans[0]):.4f} kN")

|F| = 16.1130 kN


In [6]:
# 11.5
from sympy import *

# 1. 统一单位输入 (kN, m)
P = 5
L = 6
H = 4
E = 170 * 10**6      # GPa 转换为 kPa (kN/m^2)
I = 42 * 10**-6      # mm^4 转换为 m^4
A = 206 * 10**-6     # mm^2 转换为 m^2

EI = E * I
EA = E * A

# 2. 静力学平衡计算杆 CD 的受力
# 绕铰链 A 取矩: P * (L/2) - F_C * L = 0
F_C = P / 2

# 3. 叠加法计算
# 状态 1: 假设 C 点不发生位移，简支梁跨中受集中力引起的弯曲挠度
delta_B_bending = (P * L**3) / (48 * EI)

# 状态 2: 杆 CD 弹性拉伸引起的刚体旋转
# 先计算杆 C 的伸长量 delta_C
delta_C = (F_C * H) / EA
# 因为 B 点在梁的中点，由相似三角形可知，B 点随之下降的位移是 C 点的一半
delta_B_rigid = delta_C / 2

# 4. 总挠度求和并转换单位为 mm
delta_B_total = delta_B_bending + delta_B_rigid
ans_mm = delta_B_total * 1000

print(f"|yb| = {ans_mm:.4f} mm")

|yb| = 3.2940 mm


In [15]:
# 11.6
from sympy import *

L = 5 # m
H = 3 # m
E = 153 # GPa
Iz = 49000000 # mm^4
w = 6 # kN/m
A = 207 # mm^2

L = 5 # m
H = 3 # m
E = 153*10**9 # GPa
Iz = 49 * 10**(-6) # mm^4
w = 6000 # kN/m
A = 207*10**(-6) # mm^2

T = symbols('T')

ans = solve(-(-w*L**4/(128*E*Iz)-w*L**4/(96*E*Iz)+T*L**3/(3*E*Iz))-T*H/(E*A),T)
print(ans[0]/1000)

1.61313162619699


In [16]:
# 11.7
from sympy import *

P = 12000 # kN
L = 5 # m
E = 220*10**9 # GPa
Iz = 0.000051 # m^4

w = P/L
yc = -w*L**4/(8*E*Iz) + P*L**3/(24*E*Iz) + P*L**3/(16*E*Iz)
print(yc*1000)

thetac = -w*L**3/(6*E*Iz) + P*L**2/(8*E*Iz)
print(thetac)

-2.7852049910873453
-0.0011140819964349379


In [17]:
# 11.8
from sympy import *

w = 10000 # kN/m
L = 7 # m
E = 231*10**9 # GPa
Iz = 0.000055 # m^4

M = w*L**2/24

yc = M*L**2/(2*E*Iz) - w*L**4/(128*E*Iz) - w*L**4/(96*E*Iz)
theta = M*L/(E*Iz) - w*L**3/(48*E*Iz)
print(yc*1000,theta)

4.921372819100093 0.005624426078971535


In [24]:
# 11.9
from sympy import *

L = 5 # m
E = 244*10**9 # GPa
Iz = 0.000049 # m^4
w = 11000 # kN/m

M,R = symbols('M R')

EQ1 = -w*L**4/(128*E*Iz) - w*L**4/(96*E*Iz) + R*L**3/(3*E*Iz) + M*L**2/(2*E*Iz)
EQ2 = -w*L**3/(48*E*Iz) + R*L**2/(2*E*Iz) + M*L/(E*Iz)

ans = solve((EQ1,EQ2),(M,R))
print(ans)

{M: -7161.45833333349, R: 5156.25000000005}


In [27]:
# 11.10
from sympy import *

L = 6 # m
E = 215*10**9 # GPa
Iz = 0.000055 # m^4
w = 11000 # kN/m

R = symbols('R')

y1 = -5*w*L**4/(384*E*Iz)
y2 = R*L**3/(48*E*Iz)

ans = solve(y1+y2,R)
print(ans[0]/1000)

41.2500000000002
